# Практика 1. Chain-of-Verification (CoVe)

**Цель:** проверить уже готовый ответ LLM и попытаться исправить фактические ошибки.

Источник: Dhuliawala et al., [Chain-of-Verification Reduces Hallucination in Large Language Models (ACL 2024)](https://aclanthology.org/2024.findings-acl.212/).

**Схема метода**

1. Получаем исходный ответ (в этом задании он уже дан).
2. Просим LLM составить вопросы для проверки фактов в ответе.
3. Отвечаем на **каждый вопрос отдельным запросом**, не передавая исходный ответ и предыдущие проверки.
4. Сравниваем ответы с исходным текстом и формируем исправленную версию.

> ⚠️ Это **внутренняя перепроверка** с помощью LLM, а не проверка по независимому источнику. Если модель ошибается повторно, CoVe может не найти галлюцинацию.

В демонстрации используется **OpenAI или GigaChat на выбор**. Строку `ask_llm()` впоследствии можно дополнить локальной моделью.

## 1. Установка библиотек

Запустите ячейку один раз. Она установит два SDK; далее будет использоваться только выбранный провайдер.

In [ ]:
%pip -q install openai "gigachat>=0.2.3"

## 2. Выбор модели и подключение

Поменяйте `PROVIDER` на `"openai"` или `"gigachat"`. Ключ вводится **скрыто** через `getpass()` — его не нужно сохранять в ноутбуке.

- Для OpenAI нужен **API key** OpenAI.
- Для GigaChat нужен **авторизационный ключ** GigaChat API.
- При выборе GigaChat проверьте, что выбранная модель и scope доступны вашему аккаунту.

Справка по SDK: [OpenAI Python](https://github.com/openai/openai-python), [GigaChat Python](https://github.com/ai-forever/gigachat).

In [ ]:
from getpass import getpass

# Выберите провайдера: "openai" или "gigachat"
PROVIDER = "openai"

# Названия моделей можно поменять на доступные у вас
OPENAI_MODEL = "gpt-4.1-mini"
GIGACHAT_MODEL = "GigaChat-2"

# Оставьте пустым, если отдельный сертификат для GigaChat не нужен
GIGACHAT_CA_FILE = ""

if PROVIDER == "openai":
    from openai import OpenAI

    api_key = getpass("OpenAI API key: ")
    client = OpenAI(api_key=api_key)
    MODEL = OPENAI_MODEL

elif PROVIDER == "gigachat":
    from gigachat import GigaChat

    credentials = getpass("GigaChat authorization key: ")
    settings = {
        "credentials": credentials,
        "scope": "GIGACHAT_API_PERS",
        "model": GIGACHAT_MODEL,
    }
    if GIGACHAT_CA_FILE:
        settings["ca_bundle_file"] = GIGACHAT_CA_FILE

    client = GigaChat(**settings)
    MODEL = GIGACHAT_MODEL

else:
    raise ValueError('PROVIDER должен быть "openai" или "gigachat"')

print(f"Выбран провайдер: {PROVIDER}, модель: {MODEL}")

## 3. Одна функция для обращения к любой модели

Обратите внимание: у каждого вызова только один пользовательский запрос. Мы не передаём историю переписки между шагами проверки.

In [ ]:
def ask_llm(prompt):
    """Отправить один независимый запрос выбранной LLM и вернуть текст."""
    if PROVIDER == "openai":
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content or ""

    if PROVIDER == "gigachat":
        response = client.chat.create(prompt)
        return "".join(
            part.text or ""
            for part in response.messages[0].content
            if part.text is not None
        )

## 4. Входные данные: вопрос и готовый ответ LLM

В ответе **специально допущена ошибка**. Для демонстрации не нужно сначала генерировать этот ответ.

**Задание:** попробуйте заранее определить, что верно, а что нет.

In [ ]:
QUESTION = "Кто был первым человеком в космосе, когда и на каком корабле состоялся полёт?"

DRAFT = (
    "Первым человеком в космосе стал Нил Армстронг. "
    "Он совершил полёт 12 апреля 1961 года на корабле «Восток-1»."
)

print("Исходный вопрос:", QUESTION)
print("\nОтвет, который будем проверять:\n", DRAFT)

## 5. Шаг 1 — Составить проверочные вопросы

Передаём модели исходный вопрос **и черновик**. Она должна составить вопросы для каждого ключевого факта.

Важно задавать **нейтральные вопросы**, например «Кто был первым человеком в космосе?», а не «Правда ли, что Нил Армстронг был первым человеком в космосе?».

In [ ]:
import re

questions_prompt = f"""
Ты проверяешь фактическую точность ответа LLM.

Исходный вопрос пользователя:
{QUESTION}

Ответ LLM, который нужно проверить:
{DRAFT}

Составь от 3 до 5 простых вопросов для независимой проверки
основных фактов в ответе: имён, дат, названий, событий.

Правила:
- Каждый вопрос должен проверять один конкретный факт.
- Не подсказывай предполагаемый ответ в самом вопросе.
- Не оценивай и не исправляй ответ сейчас.
- Выведи только нумерованный список вопросов: по одному на строке.
"""

raw_questions = ask_llm(questions_prompt)

# Превращаем нумерованный список модели в список строк Python
questions = []
for line in raw_questions.splitlines():
    question = re.sub(r"^\s*(?:\d+[.)]|[-*])\s*", "", line).strip()
    if "?" in question:
        questions.append(question)

questions = questions[:5]
if not questions:
    raise ValueError("Модель не вернула вопросы. Попробуйте запустить ячейку ещё раз.")

print("Проверочные вопросы:")
for i, question in enumerate(questions, start=1):
    print(f"{i}. {question}")

## 6. Шаг 2 — Независимо ответить на каждый вопрос

**Самое важное место в CoVe.** В модель отправляется **только конкретный проверочный вопрос** с общей инструкцией. Не отправляем `DRAFT`, другие вопросы или предыдущие ответы.

Это уменьшает эффект «подсматривания» в собственный неверный ответ.

In [ ]:
checks = []

for question in questions:
    answer = ask_llm(f"""
Ответь на фактологический вопрос кратко и точно.
Если не уверен, так и скажи. Не придумывай детали.

Вопрос: {question}
""")
    checks.append((question, answer))

for i, (question, answer) in enumerate(checks, start=1):
    print(f"\nПроверка {i}: {question}")
    print(f"Независимый ответ: {answer}")

## 7. Шаги 3–4 — Сравнить факты и исправить ответ

Теперь модель видит черновик и результаты независимых проверок. Попросим её:

1. Отметить, где факты **совпали**, **разошлись** или остались **неясными**.
2. Написать исправленный ответ.

Статус «совпало» **не означает доказанную истинность**: оба ответа могли оказаться одинаково неверными.

In [ ]:
from IPython.display import Markdown, display

checks_text = "\n\n".join(
    f"Проверка {i}\nВопрос: {q}\nНезависимый ответ: {a}"
    for i, (q, a) in enumerate(checks, start=1)
)

final_prompt = f"""
Сопоставь исходный ответ LLM с результатами независимой перепроверки.

Вопрос пользователя:
{QUESTION}

Исходный ответ LLM:
{DRAFT}

Результаты независимых проверок:
{checks_text}

Сделай две части:

1. "Разбор фактов" — таблица Markdown с колонками:
   Факт из исходного ответа | Статус | Комментарий.
   Статус: "совпадает с перепроверкой", "противоречит перепроверке"
   или "недостаточно данных".

2. "Исправленный ответ" — короткий, связный ответ на вопрос пользователя.

Не считай независимые ответы абсолютным доказательством.
Не выдумывай информацию при недостатке данных.
"""

report = ask_llm(final_prompt)
display(Markdown(report))

## 8. Что должно получиться

В исходном ответе специально перепутаны участники двух событий:

- **Правильно:** первым человеком в космосе был **Юрий Гагарин**, 12 апреля 1961 года, корабль **«Восток-1»**.
- **Ошибка:** в исходном ответе вместо Гагарина назван **Нил Армстронг**.

Проверьте вручную: заметила ли модель ошибку с именем? Сохранила ли верные дату и корабль? Если модель написала что-то другое, это **полезный результат эксперимента**, а не сбой занятия.

**Ограничение метода:** факты не сверялись с документами или интернетом. Это упражнение показывает именно **самопроверку модели**. Подключение внешних источников будем разбирать в следующих методах.